# Run local model answers

**Goal:** Freeze paired prompts, then run the same questions at the requested precision.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Inspect models and precision

Knowledge uses eight NF4 models. Clinical evaluation includes the same eight at NF4 and BF16. NF4 uses double quantization and BF16 computation. Check checkpoint access and available VRAM before starting a new run. SPARK requires its separate environment and the documented activation-cast adaptation.

In [ ]:
from sleepinn_study.benchmark import model_registry, run_local
models = pd.DataFrame(model_registry())
display(models.loc[models.provider.eq("huggingface"), ["name", "model_id", "revision", "knowledge_precisions", "clinical_precisions"]])

## 3. Freeze prompts once

Supply source contexts packed to eight blocks and 4,096 reference-tokenizer tokens. Perfect retrieval uses a separate source-evidence map and only applies to knowledge. Full context text is not distributed with this repository; it must come from locally licensed sources. The exact prompt templates and historical prompt hashes are included.

In [ ]:
from sleepinn_study.workflow import answer_prompt, freeze_prompts
bank = read_jsonl(ROOT / "data/final/knowledge.jsonl")
print(answer_prompt(bank[0]))
PREPARE_PROMPTS = False
if PREPARE_PROMPTS:
    freeze_prompts(bank,
        read_json(ROOT / "outputs/private/knowledge_contexts.json"), "knowledge",
        perfect_contexts=read_json(ROOT / "outputs/private/knowledge_perfect_contexts.json"))

## 4. Generate answers explicitly

Greedy generation, batch size one, 1,024 new tokens, no thinking where supported, and no automatic precision fallback or CPU offload. New outputs record model revision, prompt hashes, latency and allocated/reserved VRAM. Changing runtime or regenerating contexts creates a new experiment.

In [ ]:
RUN_MODEL = False
if RUN_MODEL:
    run_local("meta-llama/Llama-3.2-1B-Instruct", "knowledge", "NF4",
              ROOT / "outputs/local_new_001", device="cuda:0")
else:
    print("Generation disabled; saved study answers are in results/answers/.")